# Geospatial Hotspot Clustering (DBSCAN)

## Objective
Perform density-based spatial clustering (DBSCAN with Haversine metric) to detect persistent geographic flood vulnerability clusters across Mumbai.

## Input Datasets
- data/staging/staging_hotspots.csv

---

### Step 1: Haversine Distance DBSCAN Clustering

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN

base_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
df_hot = pd.read_csv(os.path.join(base_dir, "data", "staging", "staging_hotspots.csv"))

# Convert coords to radians
coords_rad = np.radians(df_hot[["latitude", "longitude"]].values)
# 6371 km Earth radius; 4.0 km epsilon
kms_per_radian = 6371.0
eps_km = 4.0
eps_rad = eps_km / kms_per_radian

db = DBSCAN(eps=eps_rad, min_samples=2, metric='haversine')
df_hot["cluster_id"] = db.fit_predict(coords_rad)

print(f"Discovered {len(set(df_hot['cluster_id'])) - (1 if -1 in df_hot['cluster_id'] else 0)} spatial clusters!")
for c in sorted(df_hot["cluster_id"].unique()):
    spots = df_hot[df_hot["cluster_id"] == c]["name"].tolist()
    print(f"Cluster {c} ({len(spots)} locations): {', '.join(spots[:4])}")

df_hot.to_csv(os.path.join(base_dir, "data", "processed", "hotspots_clustered.csv"), index=False)


Discovered 5 spatial clusters!
Cluster -1 (2 locations): Andheri Subway, Dahisar Subway
Cluster 0 (12 locations): Hindmata Cinema / Dadar TT, Gandhi Market / King's Circle, Milan Subway, Santacruz, Khar Subway
Cluster 1 (2 locations): Malad Subway, Kandivali Poisar River Culvert
Cluster 2 (2 locations): Byculla Railway Police Line, Grant Road / Nana Chowk
Cluster 3 (2 locations): Bhandup LBS Marg / Sonapur, Mulund Goregaon Link Road Jn


---
## Analytical Limitations & Assumptions
Clustering is purely spatial based on coordinate geometry; local drainage ridge barriers between nearby spots may create distinct micro-catchments.

## Key Conclusions
Identified 4 distinct high-density geographic flood clusters: Central Mithi Corridor, Island City Basin, Western Suburban Subways, and Northern Suburban Rivers.